Had help from GenAi in general to debug

In [1]:
!git clone https://github.com/gregdurrett/fp-dataset-artifacts.git
%cd fp-dataset-artifacts

Cloning into 'fp-dataset-artifacts'...
remote: Enumerating objects: 41, done.
remote: Total 41 (delta 0), reused 0 (delta 0), pack-reused 41 (from 1)
Receiving objects: 100% (41/41), 16.00 KiB | 4.00 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/fp-dataset-artifacts


In [2]:
!pip install --upgrade pip
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 73.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [4]:
!python3 run.py \
    --do_train \
    --task nli \
    --dataset snli \
    --output_dir ./snli_electra_baseline/ \
    --num_train_epochs 1

2025-12-03 17:05:42.150530: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764781542.169737    3121 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764781542.175637    3121 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764781542.190311    3121 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764781542.190335    3121 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764781542.190339    3121 computation_placer.cc:177] computation placer alr

In [5]:
!python3 run.py \
    --do_eval \
    --task nli \
    --dataset snli \
    --model ./snli_electra_baseline/ \
    --output_dir ./snli_electra_eval/


2025-12-03 17:59:54.887411: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764784794.908045   16818 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764784794.914069   16818 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764784794.930018   16818 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764784794.930044   16818 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764784794.930048   16818 computation_placer.cc:177] computation placer alr

In [14]:
%%writefile train_hypothesis.py
from datasets import load_dataset
from transformers import ElectraTokenizerFast, ElectraForSequenceClassification, Trainer, TrainingArguments

dataset = load_dataset("snli")

dataset = dataset.filter(lambda x: x["label"] != -1)

def remove_premise(example):
    example["premise"] = ""
    return example

dataset = dataset.map(remove_premise)

tokenizer = ElectraTokenizerFast.from_pretrained("google/electra-small-discriminator")

def preprocess(batch):
    return tokenizer(
        batch["premise"],
        batch["hypothesis"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

dataset = dataset.map(preprocess, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

model = ElectraForSequenceClassification.from_pretrained(
    "google/electra-small-discriminator",
    num_labels=3
)

args = TrainingArguments(
    output_dir="./snli_hypothesis/",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    logging_steps=200,
    save_total_limit=1,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"]
)

trainer.train()

trainer.save_model("./snli_hypothesis/")


Writing train_hypothesis.py


In [15]:
!python3 train_hypothesis.py

2025-12-03 19:12:12.088985: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764789132.109099   35288 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764789132.115242   35288 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764789132.130849   35288 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764789132.130878   35288 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764789132.130882   35288 computation_placer.cc:177] computation placer alr

In [ ]:
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_SILENT"] = "true"

In [18]:
#had help from GenAi to debug

!python3 - <<'EOF'
from datasets import load_dataset
from transformers import ElectraTokenizerFast, ElectraForSequenceClassification, Trainer, TrainingArguments

import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_SILENT"] = "true"

dataset = load_dataset("snli")
dataset = dataset.filter(lambda x: x["label"] != -1)

def remove_premise(example):
    example["premise"] = ""
    return example

dataset = dataset.map(remove_premise)

tokenizer = ElectraTokenizerFast.from_pretrained("google/electra-small-discriminator")

def preprocess(batch):
    return tokenizer(
        batch["premise"],
        batch["hypothesis"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

dataset = dataset.map(preprocess, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

model = ElectraForSequenceClassification.from_pretrained("./snli_hypothesis/", num_labels=3)

args = TrainingArguments(output_dir="./snli_hypothesis_eval/", per_device_eval_batch_size=32)

trainer = Trainer(model=model, args=args, eval_dataset=dataset["validation"])

print(trainer.evaluate())


/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `EOF')


Map:   0%|          | 0/9824 [00:00<?, ? examples/s]

Map:   0%|          | 0/9842 [00:00<?, ? examples/s]

Map:   0%|          | 0/549367 [00:00<?, ? examples/s]

Map:   0%|          | 0/9824 [00:00<?, ? examples/s]

Map:   0%|          | 0/9842 [00:00<?, ? examples/s]

Map:   0%|          | 0/549367 [00:00<?, ? examples/s]

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


{'eval_loss': 0.7227468490600586, 'eval_model_preparation_time': 0.0024, 'eval_runtime': 10.5927, 'eval_samples_per_second': 929.133, 'eval_steps_per_second': 29.077}


In [20]:
from datasets import load_dataset
from transformers import ElectraTokenizerFast, ElectraForSequenceClassification
import torch
import numpy as np

dataset = load_dataset("snli")
dataset = dataset.filter(lambda x: x["label"] != -1)

def remove_premise(example):
    example["premise"] = ""
    return example

dataset = dataset.map(remove_premise)

tokenizer = ElectraTokenizerFast.from_pretrained("google/electra-small-discriminator")

def preprocess(batch):
    return tokenizer(
        batch["premise"],
        batch["hypothesis"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

dataset = dataset.map(preprocess, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

model = ElectraForSequenceClassification.from_pretrained("./snli_hypothesis/").cuda()

correct = 0
total = 0

loader = torch.utils.data.DataLoader(dataset["validation"], batch_size=32)

model.eval()
with torch.no_grad():
    for batch in loader:
        input_ids = batch["input_ids"].cuda()
        attention_mask = batch["attention_mask"].cuda()
        labels = batch["labels"].cuda()

        logits = model(input_ids, attention_mask=attention_mask).logits
        preds = logits.argmax(dim=-1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
accuracy


Map:   0%|          | 0/9824 [00:00<?, ? examples/s]

Map:   0%|          | 0/9842 [00:00<?, ? examples/s]

Map:   0%|          | 0/549367 [00:00<?, ? examples/s]

Map:   0%|          | 0/9824 [00:00<?, ? examples/s]

Map:   0%|          | 0/9842 [00:00<?, ? examples/s]

Map:   0%|          | 0/549367 [00:00<?, ? examples/s]

0.6883763462710831

In [23]:
#had help from GenAi to debug

%%writefile train_debiased_snli.py
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import ElectraTokenizerFast, ElectraForSequenceClassification, Trainer, TrainingArguments

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_SILENT"] = "true"

dataset = load_dataset("snli")
dataset = dataset.filter(lambda x: x["label"] != -1)

tokenizer = ElectraTokenizerFast.from_pretrained("google/electra-small-discriminator")

def preprocess(batch):
    main_enc = tokenizer(
        batch["premise"],
        batch["hypothesis"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    bias_enc = tokenizer(
        [""] * len(batch["hypothesis"]),
        batch["hypothesis"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    return {
        "main_input_ids": main_enc["input_ids"],
        "main_attention_mask": main_enc["attention_mask"],
        "bias_input_ids": bias_enc["input_ids"],
        "bias_attention_mask": bias_enc["attention_mask"],
    }

dataset = dataset.map(preprocess, batched=True)

dataset = dataset.rename_column("label", "labels")
dataset.set_format(
    type="torch",
    columns=[
        "main_input_ids",
        "main_attention_mask",
        "bias_input_ids",
        "bias_attention_mask",
        "labels",
    ],
)

class DebiasedElectra(nn.Module):
    def __init__(self, main_model_name, bias_model_path, num_labels=3):
        super().__init__()
        self.main_model = ElectraForSequenceClassification.from_pretrained(
            main_model_name, num_labels=num_labels
        )
        self.bias_model = ElectraForSequenceClassification.from_pretrained(
            bias_model_path, num_labels=num_labels
        )
        for p in self.bias_model.parameters():
            p.requires_grad = False

        self.config = self.main_model.config
        self.num_labels = num_labels

    def forward(
        self,
        main_input_ids=None,
        main_attention_mask=None,
        bias_input_ids=None,
        bias_attention_mask=None,
        labels=None,
        **kwargs
    ):
        main_out = self.main_model(
            input_ids=main_input_ids,
            attention_mask=main_attention_mask,
            labels=None,
        )
        main_logits = main_out.logits

        with torch.no_grad():
            bias_out = self.bias_model(
                input_ids=bias_input_ids,
                attention_mask=bias_attention_mask,
                labels=None,
            )
            bias_logits = bias_out.logits

        logits = main_logits - bias_logits

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)

        return {"loss": loss, "logits": logits}

model = DebiasedElectra(
    main_model_name="google/electra-small-discriminator",
    bias_model_path="./snli_hypothesis/",
    num_labels=3,
)

args = TrainingArguments(
    output_dir="./snli_debiased/",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    logging_steps=200,
    save_total_limit=1,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
)

trainer.train()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

eval_loader = DataLoader(dataset["validation"], batch_size=32)

correct = 0
total = 0
with torch.no_grad():
    for batch in eval_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch["labels"]
        outputs = model(
            main_input_ids=batch["main_input_ids"],
            main_attention_mask=batch["main_attention_mask"],
            bias_input_ids=batch["bias_input_ids"],
            bias_attention_mask=batch["bias_attention_mask"],
        )
        preds = outputs["logits"].argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print("Debiased validation accuracy:", correct / total)


Writing train_debiased_snli.py


In [24]:
!python3 train_debiased_snli.py

2025-12-03 20:14:01.589404: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764792841.615065   51014 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764792841.623417   51014 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764792841.658705   51014 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764792841.658739   51014 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764792841.658745   51014 computation_placer.cc:177] computation placer alr